# Radial Basis Nearest Neighbor (RBNN)

- Giulia Monteiro Garrido (RA: 24010281)
- Mateus Antezana da Silva (RA: 24021000)
- Vitor Furuta da Silva (RA: 24008775)

In [49]:
import numpy as np
import pandas as pd
import seaborn as sns
from random import shuffle
from ucimlrepo import fetch_ucirepo
from typing import Sequence
from functools import wraps

In [4]:
type numero = int | float

In [10]:
def tratamento_erros(funcao):
    @wraps(funcao)
    def funcao_interna(*args, **kwargs):
        try:
            return funcao(*args, **kwargs)
        except Exception as e:
            print(f"Erro na função {funcao_interna.__name__}\nTipo: {e}")
            return None
    return funcao_interna

In [42]:
class Distancias:
    def __init__(self)->None:
        pass

    def _validar_vetores(self, A:Sequence[numero], B:Sequence[numero] | numero) -> np.asarray[float] | None:
        A = np.asarray(A, dtype=float)
        B = np.asarray(B, dtype=float)

        if isinstance(B, list) and A.shape != B.shape:
            raise ValueError("Os vetores devem ter a mesma dimensão.")

        if A.size == 0:
            raise ValueError("Os vetores não podem estar vazios.")
        return A, B

    @tratamento_erros
    def minkowski(self, A:Sequence[numero], B:Sequence[numero] | numero, p:int) -> numero | None:
        if p <= 0:
            raise ValueError("p deve ser maior que zero.")
        A, B = self._validar_vetores(A, B) 
        return sum(abs(A[i]-(B[i] if(isinstance(B, list)) else B))**p for i in range(len(A))) ** (1/p)

    @tratamento_erros
    def cosseno(self, A:Sequence[numero], B:Sequence[numero]) -> numero | None:
        if(not isinstance(B, list)):
            print("B precisa ser uma lista")
            return
        A, B = self._validar_vetores(A, B)

        produto = np.dot(A, B)
        norma_A = np.linalg.norm(A)
        norma_B = np.linalg.norm(B)

        if norma_A == 0 or norma_B == 0:
            return 0
        return float(1 - (produto/ (norma_A * norma_B)))

    @tratamento_erros
    def manhattan(self, A:Sequence[numero], B:Sequence[numero]) -> numero | None:
        A, B = self._validar_vetores(A, B)
        return sum((abs(A[i]-(B[i] if(isinstance(B, list)) else B ))) for i in range(len(A)))

    @tratamento_erros
    def euclidiana(self, A:Sequence[numero], B:Sequence[numero]) -> numero | None:
        A, B = self._validar_vetores(A, B)
        return sum(((A[i]-(B[i] if isinstance(B, list) else B))**2) for i in range(len(A)))**(1/2)

In [ ]:
def pesos(x:Sequence[numero], X_treino:Sequence[numero], sigma:numero, distancia:Distancias)->Sequence[numero]|None: # gaussiana
    if(sigma<=0):
        return f"Sigma precisa ser maior que 0, {sigma} inserido."
    distancias = np.array([
        distancia(x, xi) for xi in X_treino
    ])
    formula = 1 / (sigma * np.sqrt(2 * np.pi))
    expoente = (distancias ** 2) / (2 * sigma ** 2)
    pesos = formula * np.exp(-expoente)
    return pesos

def regressao(x:Sequence[numero], X_treino:Sequence[numero], y_treino, sigma:numero, distancia:Distancias)->numero:
    if(sigma<=0):
        return f"Sigma precisa ser maior que 0, {sigma} inserido."
    w = pesos(x, X_treino, sigma, distancia)

    if w.sum() == 0:

        distancias = np.array([distancia(x, xi) for xi in X_treino])
        return y_treino[np.argmin(distancias)]

    return np.sum(w * y_treino) / np.sum(w)

def classificacao(x, X_treino, y_treino, sigma, distancia):
    if(sigma<=0):
        return f"Sigma precisa ser maior que 0, {sigma} inserido."
    w = pesos(x, X_treino, sigma, distancia)

    if w.sum() == 0:

        distancias = np.array([distancia(x, xi) for xi in X_treino])
        return y_treino[np.argmin(distancias)]

    y_treino = np.asarray(y_treino)
    classes = np.unique(y_treino)
    votos = {c: w[y_treino == c].sum() for c in classes}
    return max(votos, key=votos.get)


In [ ]:
class RBNNRegressor:
    def __init__(self, sigma: numero, distancia:numero)->None:
        self.sigma = sigma
        self.distancia = distancia

    @tratamento_erros
    def fit(self, X_treino, y_treino)->None:
        self.X_treino = np.asarray(X_treino)
        self.y_treino = np.asarray(y_treino)

    @tratamento_erros
    def predict(self, X_teste):
        return np.array([
            regressao(x, self.X_treino, self.y_treino, self.sigma, self.distancia)
            for x in X_teste
        ])

In [44]:
class RBNNClassifier:
    def __init__(self, sigma: float, distancia)->None:
        self.sigma = sigma
        self.distancia = distancia

    @tratamento_erros
    def fit(self, X_treino:Sequence[numero], y_treino:Sequence[numero])->None:
        self.X_treino = np.asarray(X_treino)
        self.y_treino = np.asarray(y_treino)

    @tratamento_erros
    def predict(self, X_teste)->Sequence[numero]:
        return np.array([
            classificacao(x, self.X_treino, self.y_treino, self.sigma, self.distancia)
            for x in X_teste
        ])

## Carregando bases

### IRIS

- Importando/tratando

In [15]:
iris = sns.load_dataset("iris")

X_iris = iris.drop(columns="species").to_numpy()
y_iris = iris["species"].to_numpy()

print(iris.shape)          # (150, 5)
iris.head()


(150, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


#### Treino iris

- Embaralhando indices

In [102]:
indices_iris = np.arange(len(X_iris))
shuffle(indices_iris)

- Treinando iris

In [163]:
# teste mudar a porcentagem, vai ver que aumentando ele vai overfittar
# se ficar muito baixa a porcentagem, vai só chutar
treino_iris:int = int(0.1 * len(indices_iris)) #usando x% para treino
deixado_pra_teste = round(1 - (treino_iris/len(indices_iris)), 2)

idx_treino = indices_iris[:treino_iris]
idx_teste = indices_iris[treino_iris:]

x_treino, x_teste = X_iris[idx_treino], X_iris[idx_teste]

y_treino, y_teste = y_iris[idx_treino], y_iris[idx_teste]

lista_dist = [dist.euclidiana, dist.manhattan]
sigma = 50
for d in lista_dist:
    classificador = RBNNClassifier(sigma = sigma, distancia = d)

    classificador.fit(x_treino, y_treino)

    prever_iris = classificador.predict(x_teste)

    print(f"De acordo com nosso modelo, e usando a distancia {d.__func__},\
          \n\tacreditamos que os outros {deixado_pra_teste*100}% deixados para teste são:\
          \n{prever_iris[:45]}\n")

De acordo com nosso modelo, e usando a distancia <function Distancias.euclidiana at 0x000002046C61CCA0>,          
	acreditamos que os outros 90.0% deixados para teste são:          
['setosa' 'setosa' 'setosa' 'versicolor' 'setosa' 'versicolor'
 'versicolor' 'versicolor' 'setosa' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'virginica' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'setosa' 'virginica' 'versicolor'
 'setosa' 'versicolor' 'setosa' 'versicolor' 'setosa' 'virginica'
 'versicolor' 'setosa' 'versicolor' 'virginica' 'versicolor' 'versicolor'
 'setosa' 'versicolor' 'setosa' 'versicolor' 'virginica' 'versicolor'
 'setosa' 'versicolor' 'setosa' 'setosa']

De acordo com nosso modelo, e usando a distancia <function Distancias.manhattan at 0x000002046C61CB40>,          
	acreditamos que os outros 90.0% deixados para teste são:          
['setosa' 'setosa' 'setosa' 'virginica' 'setosa' 'virginica' 'versicolor'
 'versicolor' 'setosa' 'versico

### Abalone

- Importando/tratando

In [160]:
# fetch dataset 
abalone = fetch_ucirepo(id=1) 
  
# data (as pandas dataframes) 
X_abalone =  pd.DataFrame(abalone.data.features)
X_abalone = pd.get_dummies(
    X_abalone,
    columns= ['Sex'],
    dtype = int
).to_numpy()

y_abalone =  pd.DataFrame(abalone.data.targets).to_numpy().ravel()

- Embaralhando indices

In [ ]:
indices_abalone = np.arange(len(X_abalone))
shuffle(indices_abalone)

- Treinando abalone 

In [164]:
treino_abalone:int = int(0.7 * len(indices_abalone)) #usando 80% para treino
deixado_pra_teste = round(1 - (treino_abalone/len(indices_abalone)), 2)

idx_treino = indices_abalone[:treino_abalone]
idx_teste = indices_abalone[treino_abalone:]

x_treino, x_teste = X_abalone[idx_treino], X_abalone[idx_teste]

y_treino, y_teste = y_abalone[idx_treino], y_abalone[idx_teste]

sigma = 50
for d in lista_dist:
    classificador = RBNNClassifier(sigma = sigma, distancia = d)

    classificador.fit(x_treino, y_treino)

    prever_iris = classificador.predict(x_teste)

    print(f"De acordo com nosso modelo, e usando a distancia {d.__func__},\
          \n\tacreditamos que os outros {deixado_pra_teste*100}% deixados para teste são:\
          \n{prever_iris[:45]}\n")

De acordo com nosso modelo, e usando a distancia <function Distancias.euclidiana at 0x000002046C61CCA0>,          
	acreditamos que os outros 30.0% deixados para teste são:          
[9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9
 9 9 9 9 9 9 9 9]

De acordo com nosso modelo, e usando a distancia <function Distancias.manhattan at 0x000002046C61CB40>,          
	acreditamos que os outros 30.0% deixados para teste são:          
[9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 9
 9 9 9 9 9 9 9 9]

